<a href="https://colab.research.google.com/github/kangwonlee/nmisp/blob/main/50_ode/50_Attractors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In [ ]:
# This cell is for the Google Colaboratory
# https://stackoverflow.com/a/63519730
if 'google.colab' in str(get_ipython()):
  path_py = '/content/nmisp_py'

  import os
  if not os.path.exists(path_py):
    import subprocess
    subprocess.run(
        ('git', 'clone', 'https://github.com/kwlee2025cpp/nmisp_py')
    )
  assert os.path.exists(path_py)

  import sys
  sys.path.insert(0, path_py)



In [ ]:
import matplotlib.pyplot as plt
import numpy as np



In [ ]:
import ode_solver



# Numerical Simulation of Chaotic Systems: The Lorenz Attractor.<br>카오스 시스템의 수치 시뮬레이션 : 로렌츠 계



references :
* https://en.wikipedia.org/wiki/Lorenz_system
* https://behance.net/gallery/7618879/Strange-Attractors
* https://en.wikipedia.org/wiki/List_of_chaotic_maps
* https://matplotlib.org/stable/gallery/mplot3d/scatter3d.html
* https://gemini.google.com



We can use ODE solvers to calculate the trajectories of chaotic systems.<br>
ODE solver 는 카오스 시스템의 궤적을 계산하기 위해서도 사용할 수 있다.



## The Lorenz System<br>로렌츠 계



In 1963, Prof. Edward Lorenz and his colleagues developed a simplified mathematical model of atmospheric convection.<br>1963년 미국의 기상학자이자 수학자인 에드워드 로렌츠가 동료들과 함께 대기의 대류 현상을 수학적으로 단순화하여 표현하기 위해 개발한 모델이다.


The Lorenz attractor is a set of chaotic solutions; an attractor is a region in the state space where a system tends to evolve towards over time, often regardless of its initial conditions within a particular region.<br>
Lorenz Attractor 로렌츠 끌개란 그 카오스 해의 집합을 말한다. Attractor 끌개 란, 시간이 흐름에 따라 시스템이 상태 공간상 특정 영역 안에서 시작하였을 때 도달하게 되는 상태공간상의 영역 또는 평형점을 말한다.



$$
\begin{align}
    \frac{dx}{dt} & = \sigma (y - x) \\
    \frac{dy}{dt} & = x (\rho - z) - y \\
    \frac{dz}{dt} & = x y - \beta z
\end{align}
$$


The Lorenz system models a simplified scenario of a two-dimensional fluid layer uniformly warmed below and cooled from above.<br>해당 미분방정식이 나타내는 로렌츠 계는 아래에서 덥혀지고 위에서 냉각되는 유체의 움직임에 관한 것이다.



Please consider trying different values of the following parameters of the Lorenz system<br>아래 로렌츠 계 매개변수의 다른 값을 시도해 보는 것도 생각해 보자.



In [ ]:
sigma = 10.0
rho = 28.0
beta = 8.0 / 3.0



In [ ]:
def get_Lorenz_system_slope(sigma:float, rho:float, beta:float):
    '''
    Return a closure calculating slope function
    기울기 함수를 계산하는 내포 함수를 반환
    '''
    def f(t:float, xv:np.ndarray,):
        '''
        Calculate slope vector of the Lorenz system
        로렌츠 계의 기울기 벡터를 계산
        '''
        x, y, z = xv
        dx_dt = sigma * (y - x)
        dy_dt = x * (rho - z) - y
        dz_dt = x * y - beta * z

        return np.array((dx_dt, dy_dt, dz_dt))

    return f


Lorenz_attractor = get_Lorenz_system_slope(sigma=sigma, rho=rho, beta=beta)



In [ ]:
def sim_attractor(
        attractor, x_0:np.ndarray=(np.array((1, 1, 1)) * 1.0),
        sigma:float=10.0, rho:float=28.0, beta:float=(8.0/3.0),
        t_end=60.0,
        elev_deg:float=None, azim_deg:float=None, figsize=(14, 14)
    ):
    '''
    Simulate a chatoic attractor
    카오스 계 해 곡선을 시뮬레이션
    '''
    t_array = np.arange(0, t_end, 1e-3)

    t_out, x_out = ode_solver.rk4(attractor, t_array, x_0)
    x_out = np.array(x_out)
    ax = plt.figure(figsize=figsize).add_subplot(projection="3d")
    ax.view_init(elev_deg, azim_deg)
    ax.plot(x_out[:, 0], x_out[:, 1], x_out[:, 2])
    ax.plot(
        (x_out[0, 0], x_out[-1, 0]),
        (x_out[0, 1], x_out[-1, 1]),
        (x_out[0, 2], x_out[-1, 2]),
        'o'
    )
    ax.text(x_out[0, 0], x_out[0, 1], x_out[0, 2], 'start')
    ax.text(x_out[-1, 0], x_out[-1, 1], x_out[-1, 2], 'end')

    ax.set_xlabel('rate of convection')
    ax.set_ylabel('horizontal temperature variation')
    ax.set_zlabel('vertical temperature variation')
    ax.grid(True)

    return ax



In [ ]:
%time ax = sim_attractor(Lorenz_attractor, (np.array((1, 1, 1)) * 1.0))



Now let's slightly change the initial condition. How different is the final point this time?<br>이제 초기 조건을 살짝 바꾸어 보자. 마지막 점의 위치는 얼마 정도 달라졌는가?



In [ ]:
%time ax = sim_attractor(Lorenz_attractor, (np.array((1, 1, 1.001)) * 1.0))



## How fast do two nearby trajectories drift apart? <br>아주 가까이서 출발한 두 궤적은 얼마나 빨리 멀어질까?


The two runs above started only 0.001 apart in the z-direction but ended at very different points. Let's measure the distance between the two trajectories as a function of time.<br>위 두 시뮬레이션은 z 방향으로 0.001 만큼만 차이가 났을 뿐인데 마지막 위치가 크게 달라졌다. 두 궤적의 거리를 시간에 대해 그려서 얼마나 빨리 멀어지는지 확인해 보자.


In [ ]:
t_array = np.arange(0, 30, 1e-3)

_, traj_A = ode_solver.rk4(Lorenz_attractor, t_array, np.array([1.0, 1.0, 1.0]))
_, traj_B = ode_solver.rk4(Lorenz_attractor, t_array, np.array([1.0, 1.0, 1.001]))

traj_A = np.array(traj_A)
traj_B = np.array(traj_B)

distance = np.linalg.norm(traj_A - traj_B, axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(t_array, distance)
ax.set_xlabel('time')
ax.set_ylabel('|trajectory A - trajectory B|  (log scale)')
ax.set_title('Two trajectories started 0.001 apart')
ax.grid(True)
plt.show()


The distance shoots up nearly straight on the log-scale plot — that means it grows **exponentially**. Tiny differences in the starting point blow up fast. This is called *sensitive dependence on initial conditions*, the technical name for the famous 'butterfly effect.'<br>로그 눈금에서 거의 직선으로 가파르게 올라간다 → 거리가 **지수적으로** 커진다는 뜻이다. 출발점의 아주 작은 차이가 빠르게 커진다. 이 현상을 *초기 조건에 대한 민감한 의존성* 이라고 부른다 — 흔히 '나비효과' 라고도 한다.


## Play with the parameters: σ, ρ, β <br>매개변수를 직접 바꾸어 보자


Move the sliders to tune the Lorenz system parameters and watch the shape of the attractor change.<br>슬라이더를 움직여서 매개변수 값을 바꾸어 가며 끌개의 모양이 어떻게 달라지는지 확인해 보자.

* ρ < 1 → trajectories die to the origin<br>ρ < 1 일 때, 궤적은 원점으로 수렴한다.
* 1 ≤ ρ ≲ 13.9 → two stable fixed points (no chaos)<br>1 ≤ ρ ≲ 13.9 일 때, 두 개의 안정 평형점에 도달한다 (카오스 없음).
* ρ ≳ 24.74 → chaotic butterfly<br>ρ ≳ 24.74 일 때, 카오스 끌개 (나비 모양) 가 나타난다.


In [ ]:
# Slider uses scipy.integrate.solve_ivp for responsiveness.
# Each slider move re-runs a short simulation.
from scipy.integrate import solve_ivp
from ipywidgets import interact, FloatSlider

def plot_lorenz(sigma=10.0, rho=28.0, beta=8.0/3.0):
    def f(t, xv):
        x, y, z = xv
        return [sigma * (y - x), x * (rho - z) - y, x * y - beta * z]

    sol = solve_ivp(
        f, (0, 25), [1.0, 1.0, 1.0],
        method='RK45', rtol=1e-6,
        t_eval=np.linspace(0, 25, 5000),
    )

    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(projection='3d')
    ax.plot(sol.y[0], sol.y[1], sol.y[2], linewidth=0.5)
    ax.set_title(f'sigma={sigma:.1f},  rho={rho:.1f},  beta={beta:.2f}')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_zlabel('z')
    plt.show()

interact(
    plot_lorenz,
    sigma=FloatSlider(min=1.0,  max=20.0,  step=0.5,  value=10.0),
    rho  =FloatSlider(min=0.5,  max=99.96, step=0.5,  value=28.0),
    beta =FloatSlider(min=0.5,  max=5.0,   step=0.1,  value=8.0/3.0),
);


To analyze such nonlinear systems, can numerical methods such as RK4 be helpful?<br>이러한 비선형 시스템 해석에 RK4와 같은 수치 해법이 도움이 될 수 있다고 생각하는가?

